In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/Crawler")
DATA_DIR = BASE_DIR / "data"
H_DIR = DATA_DIR / "h"

H_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("H_DIR:", H_DIR)

BASE_DIR: /content/drive/MyDrive/Crawler
DATA_DIR: /content/drive/MyDrive/Crawler/data
H_DIR: /content/drive/MyDrive/Crawler/data/h


In [3]:
# 입력 파일 경로
h_candidates_path = H_DIR / "h_kakao_restaurant_candidates.csv"

# 출력 파일 경로
h_status_out_path = H_DIR / "h_02_review_collection_status.csv"
h_raw_reviews_out_path = H_DIR / "h_02_raw_reviews_manual.csv"

print("입력:", h_candidates_path)
print("상태 CSV 출력:", h_status_out_path)
print("리뷰 CSV 출력:", h_raw_reviews_out_path)

입력: /content/drive/MyDrive/Crawler/data/h/h_kakao_restaurant_candidates.csv
상태 CSV 출력: /content/drive/MyDrive/Crawler/data/h/h_02_review_collection_status.csv
리뷰 CSV 출력: /content/drive/MyDrive/Crawler/data/h/h_02_raw_reviews_manual.csv


In [4]:
# 인사캠 카카오 식당 후보 파일 읽기
df_h = pd.read_csv(h_candidates_path)

print("인사캠 후보 수:", len(df_h))
print("컬럼:", df_h.columns.tolist())

display(df_h.head())

인사캠 후보 수: 45
컬럼: ['restaurant_temp_id', 'campus_area', 'center_query', 'radius_m', 'kakao_place_id', 'name', 'category_name', 'address', 'road_address', 'longitude', 'latitude', 'phone', 'kakao_place_url', 'distance_m']


,restaurant_temp_id,campus_area,center_query,radius_m,kakao_place_id,name,category_name,address,road_address,longitude,latitude,phone,kakao_place_url,distance_m
0,1,skku_humanities_front_gate,성균관대학교 인문사회과학캠퍼스 정문,300,168487509,인생건어물맥주 대학로점,"음식점 > 술집 > 호프,요리주점 > 인생건어물맥주",서울 종로구 명륜2가 122,서울 종로구 성균관로 26,126.997319,37.585003,NaN,http://place.map.kakao.com/168487509,42.4
1,2,skku_humanities_front_gate,성균관대학교 인문사회과학캠퍼스 정문,300,11634881,한솥도시락 성균관대앞점,음식점 > 도시락 > 한솥도시락,서울 종로구 명륜2가 122,서울 종로구 성균관로 26,126.997324,37.584999,02-744-0762,http://place.map.kakao.com/11634881,42.7
2,3,skku_humanities_front_gate,성균관대학교 인문사회과학캠퍼스 정문,300,22016023,맥덕스,음식점 > 양식,서울 종로구 명륜2가 135-3,서울 종로구 성균관로 25-9,126.997020,37.584635,070-8623-6397,http://place.map.kakao.com/22016023,43.6
3,4,skku_humanities_front_gate,성균관대학교 인문사회과학캠퍼스 정문,300,338935651,명륜진사갈비 서울성균관대점,"음식점 > 한식 > 육류,고기 > 갈비 > 명륜진사갈비",서울 종로구 명륜3가 53,서울 종로구 성균관로 31,126.996753,37.585392,02-747-9279,http://place.map.kakao.com/338935651,44.2
4,5,skku_humanities_front_gate,성균관대학교 인문사회과학캠퍼스 정문,300,122821905,물결식당,음식점 > 퓨전요리 > 퓨전한식,서울 종로구 명륜2가 135-3,서울 종로구 성균관로 25-9,126.997018,37.584619,NaN,http://place.map.kakao.com/122821905,45.3


In [5]:
# 기존 자과캠 형식에 맞춘 인사캠 review_collection_status 빈 파일 생성

TARGET_REVIEW_COUNT = 50

required_cols = [
    "kakao_place_id",
    "name",
    "kakao_place_url",
]

missing = [c for c in required_cols if c not in df_h.columns]
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

df_h_status = pd.DataFrame({
    "restaurant_external_id": df_h["kakao_place_id"].astype(str).apply(lambda x: f"kakao_{x}"),
    "restaurant_name": df_h["name"],
    "kakao_place_id": df_h["kakao_place_id"].astype(str),
    "kakao_place_url": df_h["kakao_place_url"],
    "collected_review_count": 0,
    "status": "not_started",
})

# 중복 제거
df_h_status = df_h_status.drop_duplicates(subset=["restaurant_external_id"]).reset_index(drop=True)

df_h_status.to_csv(h_status_out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", h_status_out_path)
print("행 수:", len(df_h_status))

display(df_h_status.head(10))

저장 완료: /content/drive/MyDrive/Crawler/data/h/h_02_review_collection_status.csv
행 수: 45


,restaurant_external_id,restaurant_name,kakao_place_id,kakao_place_url,collected_review_count,status
0,kakao_168487509,인생건어물맥주 대학로점,168487509,http://place.map.kakao.com/168487509,0,not_started
1,kakao_11634881,한솥도시락 성균관대앞점,11634881,http://place.map.kakao.com/11634881,0,not_started
2,kakao_22016023,맥덕스,22016023,http://place.map.kakao.com/22016023,0,not_started
3,kakao_338935651,명륜진사갈비 서울성균관대점,338935651,http://place.map.kakao.com/338935651,0,not_started
4,kakao_122821905,물결식당,122821905,http://place.map.kakao.com/122821905,0,not_started
5,kakao_25770908,피자헤븐 종로성대점,25770908,http://place.map.kakao.com/25770908,0,not_started
6,kakao_1869234515,맘스터치 피자앤치킨 종로성대점,1869234515,http://place.map.kakao.com/1869234515,0,not_started
7,kakao_483030266,명분,483030266,http://place.map.kakao.com/483030266,0,not_started
8,kakao_633176459,집시포차 혜화점,633176459,http://place.map.kakao.com/633176459,0,not_started
9,kakao_11520636,페르시안궁전,11520636,http://place.map.kakao.com/11520636,0,not_started


In [6]:
# 기존 자과캠 raw_reviews_manual 형식에 맞춘 인사캠 빈 리뷰 파일 생성
# 식당당 50개 행을 미리 생성하고 review_text만 비워둠

rows = []

for _, r in df_h_status.iterrows():
    restaurant_external_id = r["restaurant_external_id"]
    restaurant_name = r["restaurant_name"]

    for i in range(1, TARGET_REVIEW_COUNT + 1):
        rows.append({
            "review_external_id": f"{restaurant_external_id}_naver_{i:03d}",
            "restaurant_external_id": restaurant_external_id,
            "restaurant_name": restaurant_name,
            "review_index": i,
            "review_text": "",
        })

df_h_raw = pd.DataFrame(rows)

df_h_raw.to_csv(h_raw_reviews_out_path, index=False, encoding="utf-8-sig")

print("저장 완료:", h_raw_reviews_out_path)
print("행 수:", len(df_h_raw))
print("예상 행 수:", len(df_h_status) * TARGET_REVIEW_COUNT)

display(df_h_raw.head(60))

저장 완료: /content/drive/MyDrive/Crawler/data/h/h_02_raw_reviews_manual.csv
행 수: 2250
예상 행 수: 2250


,review_external_id,restaurant_external_id,restaurant_name,review_index,review_text
0,kakao_168487509_naver_001,kakao_168487509,인생건어물맥주 대학로점,1,
1,kakao_168487509_naver_002,kakao_168487509,인생건어물맥주 대학로점,2,
2,kakao_168487509_naver_003,kakao_168487509,인생건어물맥주 대학로점,3,
3,kakao_168487509_naver_004,kakao_168487509,인생건어물맥주 대학로점,4,
4,kakao_168487509_naver_005,kakao_168487509,인생건어물맥주 대학로점,5,
5,kakao_168487509_naver_006,kakao_168487509,인생건어물맥주 대학로점,6,
6,kakao_168487509_naver_007,kakao_168487509,인생건어물맥주 대학로점,7,
7,kakao_168487509_naver_008,kakao_168487509,인생건어물맥주 대학로점,8,
8,kakao_168487509_naver_009,kakao_168487509,인생건어물맥주 대학로점,9,
9,kakao_168487509_naver_010,kakao_168487509,인생건어물맥주 대학로점,10,
